<a href="https://colab.research.google.com/github/thatcherty/ML-Algo-Selection/blob/Log-Reg/ML_Algo_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Algo Selection

In [1]:
# Common
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time
from pandas.errors import ParserError

# requires pip install ucimlrepo
!pip install ucimlrepo
from ucimlrepo import fetch_ucirepo


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score, accuracy_score

# additional for Neural Network
# include MLP or CNN
import tensorflow as tf
from tensorflow import keras

dataset = fetch_ucirepo(id=1)
print(dataset.metadata)


{'uci_id': 1, 'name': 'Abalone', 'repository_url': 'https://archive.ics.uci.edu/dataset/1/abalone', 'data_url': 'https://archive.ics.uci.edu/static/public/1/data.csv', 'abstract': 'Predict the age of abalone from physical measurements', 'area': 'Biology', 'tasks': ['Classification', 'Regression'], 'characteristics': ['Tabular'], 'num_instances': 4177, 'num_features': 8, 'feature_types': ['Categorical', 'Integer', 'Real'], 'demographics': [], 'target_col': ['Rings'], 'index_col': None, 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 1994, 'last_updated': 'Mon Aug 28 2023', 'dataset_doi': '10.24432/C55C7W', 'creators': ['Warwick Nash', 'Tracy Sellers', 'Simon Talbot', 'Andrew Cawthorn', 'Wes Ford'], 'intro_paper': None, 'additional_info': {'summary': 'Predicting the age of abalone from physical measurements.  The age of abalone is determined by cutting the shell through the cone, staining it, and counting the number of rings through a microscope -- 

In [2]:
import argparse
import re
from pathlib import Path
from urllib.parse import parse_qsl, urlencode, urljoin, urlparse, urlunparse

import requests
from bs4 import BeautifulSoup


BASE_URL = (
    "https://archive.ics.uci.edu/datasets"
    "?Task=Classification"
    "&Types=Multivariate"
    "&Types=Tabular"
    "&Python=true"
    "&skip=0"
    "&take=10"
    "&sort=desc"
    "&orderBy=NumHits"
    "&search="
)

USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0 Safari/537.36"
)


def build_page_url(base_url: str, *, skip: int, take: int) -> str:
    parsed = urlparse(base_url)

    # Keep repeated query params like Types=...
    params = parse_qsl(parsed.query, keep_blank_values=True)

    new_params = []
    skip_set = False
    take_set = False

    for key, value in params:
        if key == "skip":
            new_params.append(("skip", str(skip)))
            skip_set = True
        elif key == "take":
            new_params.append(("take", str(take)))
            take_set = True
        else:
            new_params.append((key, value))

    if not skip_set:
        new_params.append(("skip", str(skip)))
    if not take_set:
        new_params.append(("take", str(take)))

    new_query = urlencode(new_params, doseq=True)
    return urlunparse(parsed._replace(query=new_query))


def extract_total_count(page_text: str) -> int | None:
    match = re.search(r"\b\d+\s+to\s+\d+\s+of\s+(\d+)\b", page_text)
    return int(match.group(1)) if match else None


def parse_dataset_links(html: str, page_url: str) -> list[int]:
    soup = BeautifulSoup(html, "html.parser")

    anchors = soup.select("a[href*='/dataset/']")
    dataset_ids = []
    seen = set()

    for a in anchors:
        href = (a.get("href") or "").strip()
        if not href:
            continue

        full_url = urljoin(page_url, href)

        match = re.search(r"/dataset/(\d+)", full_url)
        if not match:
            continue

        dataset_id = int(match.group(1))

        if dataset_id in seen:
            continue
        seen.add(dataset_id)
        dataset_ids.append(dataset_id)

    return dataset_ids


def scrape_all_datasets(base_url: str, *, take: int = 25, timeout: int = 30) -> list[int]:
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    all_ids = []
    seen_ids = set()
    skip = 0
    total_count = None

    while True:
        page_url = build_page_url(base_url, skip=skip, take=take)

        response = session.get(page_url, timeout=timeout)
        response.raise_for_status()

        html = response.text
        text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)

        if total_count is None:
            total_count = extract_total_count(text)
            print(f"Total count reported: {total_count}")

        page_ids = parse_dataset_links(html, page_url)
        new_ids = [dataset_id for dataset_id in page_ids if dataset_id not in seen_ids]

        if not new_ids:
            break

        for dataset_id in new_ids:
            seen_ids.add(dataset_id)
            all_ids.append(dataset_id)

        if total_count is not None and len(all_ids) >= total_count:
            break

        if len(page_ids) < take:
            break

        skip += take

    return all_ids


def get_data() -> list[int]:
    parser = argparse.ArgumentParser(
        description="Scrape UCI dataset IDs matching the filtered URL."
    )
    parser.add_argument("--url", default=BASE_URL, help="Filtered UCI listing URL to scrape.")
    parser.add_argument(
        "--take",
        type=int,
        default=25,
        help="Rows requested per page while paginating."
    )
    parser.add_argument(
        "--timeout",
        type=int,
        default=30,
        help="HTTP timeout in seconds."
    )
    args = parser.parse_args([])

    ids = scrape_all_datasets(args.url, take=args.take, timeout=args.timeout)

    if not ids:
        raise SystemExit("No dataset IDs were found. The page structure may have changed.")

    return ids

In [3]:
# extract ids to exclude based on characertistics

exclude_types: list[str] = ["Time-Series", "Image", "Sequential", "Spatiotemporal", "Text", "Other"]

ids = get_data()
print(ids)

excluded_ids = []
final_ids = []

for dataset_id in ids:
    try:
        dataset = fetch_ucirepo(id=dataset_id)
        y = dataset.data.targets

        if y is None or y.empty:
          print(f"Skipping dataset {dataset_id} ({dataset.metadata['name']}) due to no target variables.")
          excluded_ids.append(dataset_id)
          continue

        # get min class size
        min_class_size = y.value_counts().min()

        if min_class_size < 2:
          print(f"Class size < 2 for {dataset_id}")
          excluded_ids.append(dataset_id)
          continue

        final_ids.append(dataset_id)
    except ParserError as e:
        print(f"Skipping {dataset_id} due to parse error: {e}")
        excluded_ids.append(dataset_id)
        continue
    except Exception as e:
        print(f"Skipping {dataset_id} due to fetch error: {e}")
        excluded_ids.append(dataset_id)
        continue

    characteristics = dataset.metadata.get("characteristics") or []

    if any(exclude in characteristics for exclude in exclude_types):
        print(f"Skipping {dataset_id} due to characteristics: {characteristics}")
        excluded_ids.append(dataset_id)
        continue

print("Failed dataset count:", len(excluded_ids))



Total count reported: 167
[53, 45, 186, 222, 17, 2, 320, 352, 109, 19, 697, 350, 144, 73, 544, 1, 891, 94, 468, 159, 15, 296, 14, 519, 602, 27, 20, 336, 242, 601, 292, 174, 327, 80, 42, 545, 267, 856, 31, 111, 555, 967, 62, 332, 597, 863, 373, 59, 529, 579, 936, 942, 46, 848, 563, 890, 52, 383, 878, 225, 572, 850, 857, 887, 445, 158, 151, 101, 571, 145, 938, 484, 915, 880, 864, 365, 547, 110, 759, 193, 244, 732, 33, 763, 105, 143, 76, 451, 176, 603, 198, 12, 264, 16, 379, 471, 760, 212, 39, 461, 172, 467, 90, 536, 503, 357, 50, 419, 47, 161, 117, 81, 485, 911, 58, 342, 537, 827, 30, 329, 582, 26, 40, 799, 247, 43, 565, 149, 146, 270, 13, 38, 372, 148, 913, 367, 257, 277, 713, 728, 300, 54, 28, 83, 722, 22, 184, 567, 78, 107, 95, 63, 755, 3, 70, 75, 44, 23, 91, 74, 32, 96, 82, 18, 88, 8, 147]
Class size < 2 for 320
Skipping dataset 352 (Online Retail) due to no target variables.
Class size < 2 for 1


/usr/local/lib/python3.12/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Class size < 2 for 242
Class size < 2 for 601


/usr/local/lib/python3.12/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (0,5,6,12,14,15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


Skipping dataset 555 (Apartment for Rent Classified) due to no target variables.
Class size < 2 for 332
Class size < 2 for 597
Class size < 2 for 373
Class size < 2 for 579
Skipping 942 due to characteristics: ['Tabular', 'Sequential', 'Multivariate']
Skipping dataset 383 (Cervical Cancer (Risk Factors)) due to no target variables.
Class size < 2 for 445
Class size < 2 for 938
Skipping dataset 484 (Travel Reviews) due to no target variables.
Skipping 864 due to characteristics: ['Multivariate', 'Time-Series']
Class size < 2 for 547
Class size < 2 for 193
Skipping 763 due to characteristics: ['Tabular', 'Multivariate', 'Other']
Skipping 264 due to characteristics: ['Multivariate', 'Sequential', 'Time-Series']
Class size < 2 for 471
Skipping dataset 760 (Multivariate Gait Data) due to no target variables.
Skipping dataset 461 (Drug Reviews (Druglib.com)) due to no target variables.
Skipping 172 due to characteristics: ['Multivariate', 'Sequential', 'Time-Series']
Skipping dataset 467 (St

In [4]:
ids = [53, 45, 186, 17, 222, 2, 109, 19, 697, 350, 144, 73, 544, 891, 94, 468, 159, 15, 296, 14, 519, 602, 27, 20, 336, 292, 174, 327, 80, 42, 545, 267, 856, 31, 111, 967, 62, 863, 59, 529, 936, 46, 848, 563, 890, 52, 878, 225, 572, 850, 857, 887, 158, 151, 101, 571, 145, 915, 880, 365, 110, 759, 244, 732, 33, 105, 76, 143, 451, 176, 603, 198, 12, 16, 379, 212, 39, 50, 503, 47, 419, 161, 117, 81, 58, 342, 537, 827, 30, 329, 582, 43, 26, 565, 146, 13, 372, 148, 367, 257, 277, 300, 54, 28, 728, 722, 22, 184, 78, 107, 95, 63, 3, 70, 44, 75, 91, 23, 74, 32, 96, 88, 147]

print(len(ids))

datasets = []

for dataset_id in ids:
  datasets.append(fetch_ucirepo(id=dataset_id))

123


/usr/local/lib/python3.12/dist-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


In [7]:
# get model features
def extract_features(dataset):
  # Get dataset features
  name = dataset.metadata['name']
  feature_count = dataset.metadata['num_features']
  instance_count = dataset.metadata['num_instances']
  missing_values = dataset.metadata['has_missing_values']

  #p/n ratio: if high, logistic regression more likely
  feature_instance_ratio = feature_count / instance_count

  #identifying column types: (numeric or binary)
  num_X = X.select_dtypes(include=[np.number])
  num_feature_count = num_X.shape[1]

  #avg correlation: if high, decision tree more likely
  #redundancy check
  avg_corr = 0

  if num_feature_count > 1:
    corr_matrix = num_X.corr().abs()

    #sum, then subtract diagonals and average everything else
    off_diag = corr_matrix.values.sum() - num_feature_count
    denom = num_feature_count**2 - num_feature_count
    avg_corr = off_diag / denom


  #cat ratio: if high, decision tree more likely
  if feature_count > 0:
    cat_ratio = (feature_count -  num_feature_count)/feature_count

  else:
    cat_ratio = 0

  #avg skewness: if high, decision tree more likely
  avg_skew = 0
  if num_feature_count > 0:
    avg_skew = num_X.skew().abs().mean()


  new_row = {'Dataset': name, 'Feature Count': feature_count, 'Instances': instance_count, 'Missing Values': missing_values, 'Average Correlation:':avg_corr, 'Average Skew: ':avg_skew}

  return new_row

# preprocessing
def build_preprocessor(X: pd.DataFrame):
    # Detect column types
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [col for col in X.columns if col not in numeric_cols]

    # Numeric preprocessing
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Categorical preprocessing
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
    ])

    # Combine both
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

    return preprocessor


# simulate train and test
def run_simulation(X, y, model, name, log_all = 0, log_conf = 0) -> float:
    # preprocessor will determine what preprocessing to perform on what columns
    preprocessor = build_preprocessor(X)

    # preprocess data and select
    pipe = Pipeline([('preprocessor', preprocessor), (name, model)])


    # split data into 5 sections
    # select 4 of 5 for train and last for test
    # do this selection 5 times
    min_class_size = y.value_counts().min()

    if min_class_size < 2:
        print("Class size < 2")

    n_splits = min(5, min_class_size)

    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=0)

    # fits model on each of the 5 train/test splits
    # reports all accuracy and f1 for the 5 splits
    results = cross_validate(
        pipe,
        X,
        y,
        cv=cv,
        scoring=["f1_macro"],
        return_train_score=False
    )

    # extract results
    f1_scores = results["test_f1_macro"]

    # get mean and std f1 among 5 splits
    # used to determine confidence score
    mean_f1 = np.mean(f1_scores)
    std_f1 = np.std(f1_scores)

    # determine confidence score
    # high f1 score and low std is preferred
    confidence_score = mean_f1 - std_f1

    # optional logging
    if log_all:
      print(f"{name} {dataset.metadata["uci_id"]} mean F1: {mean_f1:.4f}")
      print(f"{name} {dataset.metadata["uci_id"]} std F1: {std_f1:.4f}")

    if log_all or log_conf:
      print(f"{name} {dataset.metadata["uci_id"]} confidence score (mean F1 - std F1): {confidence_score:.4f}")

    return confidence_score

# determine best model
def get_best_model(model_scores):
  best_model = max(model_scores, key=model_scores.get)

  ties = [model for model, score in model_scores.items() if score == model_scores[best_model]]

  # choose simpler model of tied models
  if len(ties) > 1:
    preference = ["Logistic", "Decision Tree", "Neural Network"]
    for model in preference:
      if model in ties:
        best_model = model
        break
  else:
    best_model = ties[0]

  return best_model, model_scores[best_model]

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

# exclude convergence warnings for clearer logging
if True:
  warnings.filterwarnings("ignore", category=ConvergenceWarning)

count = 0

test_log = 1
test_nn = 1
test_tree = 1

# Create final dataset obj
algo_selection = pd.DataFrame({"Dataset": [], "Feature Count": [], "Instances": [], "Missing Values": [], "Recommended Algorithm": [], "Expected Confidence": []})
algo_selection["Recommended Algorithm"] = algo_selection["Recommended Algorithm"].astype("string")

for dataset in datasets:
  if count == 25:
    break
  count += 1

  model_scores = {}

  # Get the targets for the current dataset
  current_dataset_targets_df = dataset.data.targets

  # This should be done by exclusion previously, but keep just in case
  # Skip if no target variables or if the target DataFrame is empty
  if current_dataset_targets_df is None or current_dataset_targets_df.empty:
    print(f"Skipping dataset {dataset.metadata['uci_id']} ({dataset.metadata['name']}) due to no target variables.")
    continue

  # Get the list of target column names to iterate over
  targets_to_process_names = current_dataset_targets_df.columns.tolist()

  # model assessment loop to account for multiple target var
  for target_col_name in targets_to_process_names:
    X = dataset.data.features
    y = current_dataset_targets_df[target_col_name]

    # extract metadata dataset features
    new_row_df = pd.DataFrame([extract_features(dataset)])

    # Add features to final dataset obj
    algo_selection = pd.concat([algo_selection, new_row_df], ignore_index=True)

    if test_log:
      # define model
      name = "Logistic Regression"
      model = LogisticRegression(
          #regularization strength
          C=1.0,

          #regularitaion type
          penalty ='l1',

          #optimization
          solver = 'saga',

          #convergence
          max_iter=1000
      )

      # run simulation
      # this handles preprocessing and data splitting
      model_scores["Logistic"] = run_simulation(X, y, model, name, log_conf=1)

    if test_nn:
      # define model
      name = "Neural Network"
      model = MLPClassifier()

      # run simulation
      # this handles preprocessing and data splitting
      model_scores["Neural Network"] = run_simulation(X, y, model, name, log_conf=1)

    if test_tree:
      # define model
      name = "Tree"
      model = DecisionTreeClassifier(criterion = "entropy", random_state=0)

      # run simulation
      # this handles preprocessing and data splitting
      model_scores["Decision Tree"] = run_simulation(X, y, model, name, log_conf=1)


    # Determine Recommended Algorithm
    best_model, confidence = get_best_model(model_scores)
    algo_selection.loc[algo_selection["Dataset"] == dataset.metadata["name"], "Recommended Algorithm"] = best_model
    algo_selection.loc[algo_selection["Dataset"] == dataset.metadata["name"], "Expected Confidence"] = confidence
    print(f"Recommended Algorithm for {dataset.metadata["uci_id"]}: {best_model} with confidence {confidence}")

Logistic Regression 53 confidence score (mean F1 - std F1): 0.9466
Neural Network 53 confidence score (mean F1 - std F1): 0.9265
Tree 53 confidence score (mean F1 - std F1): 0.8882
Recommended Algorithm for 53: Logistic with confidence 0.9466332497911445
Logistic Regression 45 confidence score (mean F1 - std F1): 0.2223
Neural Network 45 confidence score (mean F1 - std F1): 0.2441
Tree 45 confidence score (mean F1 - std F1): 0.2562
Recommended Algorithm for 45: Decision Tree with confidence 0.2561532312889037
Logistic Regression 186 confidence score (mean F1 - std F1): 0.2071
Neural Network 186 confidence score (mean F1 - std F1): 0.2604
Tree 186 confidence score (mean F1 - std F1): 0.3400
Recommended Algorithm for 186: Decision Tree with confidence 0.33998965946207427
Logistic Regression 17 confidence score (mean F1 - std F1): 0.9547
Neural Network 17 confidence score (mean F1 - std F1): 0.9628
Tree 17 confidence score (mean F1 - std F1): 0.9181
Recommended Algorithm for 17: Neural Ne